### Imports


In [ ]:
%run /data-analysts/utils/ 

In [ ]:
//imports
import java.sql.Date
import java.time.LocalDate
import java.util.Date
import java.util.Calendar
import java.text.SimpleDateFormat
import org.apache.spark.sql.functions
import org.apache.spark.sql.functions._
import org.apache.spark.sql.types._
import org.apache.spark.sql.expressions.{Window, UserDefinedFunction}
import org.apache.spark.sql.{Column, DataFrame, Dataset}
import org.apache.spark.sql.types.{IntegerType, StringType, StructField, StructType, DoubleType};
import spark.implicits._
import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions._

In [ ]:
%run "Users/jorge.espinosa@nubank.com.br/ImportsScalae2"

### Data Loading
Challenge 4 — MTU Fraud Policy. Load four **Parquet** files (full dataset) from `/Volumes/usr/luisdominguez/hackaton/`.

In [ ]:
%python
import pandas as pd

base = "/Volumes/usr/luisdominguez/hackaton"

files = {
    "transactions":  f"{base}/transactions.parquet",
    "customer_mtu":  f"{base}/customer_mtu.parquet",
    "policy_events": f"{base}/policy_events.parquet",
    "scam_reports":  f"{base}/scam_reports.parquet",
}

for name, path in files.items():
    pdf = pd.read_parquet(path)
    # Truncate nanosecond timestamps to microseconds for Spark compatibility
    for col in pdf.select_dtypes(include=["datetime64[ns]", "datetime64[ns, UTC]"]).columns:
        pdf[col] = pdf[col].dt.floor("us")
    sdf = spark.createDataFrame(pdf)
    sdf.createOrReplaceTempView(name)
    print(f"{name}: {sdf.count()} rows, {len(sdf.columns)} cols")

In [ ]:
// Pick up the temp views created in Python
val transactions = spark.table("transactions")
val customerMtu  = spark.table("customer_mtu")
val policyEvents = spark.table("policy_events")
val scamReports  = spark.table("scam_reports")

println(s"transactions:   ${transactions.count()} rows")
println(s"customer_mtu:   ${customerMtu.count()} rows")
println(s"policy_events:  ${policyEvents.count()} rows")
println(s"scam_reports:   ${scamReports.count()} rows")

In [ ]:
println("=== transactions ===")
transactions.printSchema()
println("\n=== customer_mtu ===")
customerMtu.printSchema()
println("\n=== policy_events ===")
policyEvents.printSchema()
println("\n=== scam_reports ===")
scamReports.printSchema()

### Joins & Feature Engineering
Build the master transaction-level DataFrame with customer attributes, policy events, scam labels, and computed `mtu_ratio`.

In [ ]:
// -- 1. Enrich transactions with customer attributes
val txnCust = transactions.join(customerMtu, Seq("customer_id"), "left")

// -- 2. Left-join policy events (only ~4% of txns have a policy event)
val txnCustPolicy = txnCust.join(
  policyEvents.drop("event_id"),  // drop PK to keep tidy
  Seq("txn_id"), "left"
)

// -- 3. Left-join scam reports
val master = txnCustPolicy.join(
  scamReports.drop("report_id", "reported_ts", "report_channel"),
  Seq("txn_id"), "left"
)
  // -- 4. Compute mtu_ratio  =  (mtd_volume_before + amount) / mtu_declared
  .withColumn("mtu_ratio",
    when($"mtu_declared_mxn" > 0,
      ($"mtd_volume_before_mxn" + $"amount_mxn") / $"mtu_declared_mxn"
    ).otherwise(lit(null))
  )
  // -- 5. Geo mismatch flag
  .withColumn("geo_mismatch", $"geo_state" =!= $"home_state")
  // -- 6. Ticket anomaly: how many multiples of the customer's avg ticket
  .withColumn("ticket_ratio",
    when($"avg_ticket_90d_mxn" > 0, $"amount_mxn" / $"avg_ticket_90d_mxn").otherwise(lit(null))
  )
  // -- 7. Declared vs observed MTU gap
  .withColumn("mtu_gap_ratio",
    when($"mtu_observed_p95_mxn" > 0,
      $"mtu_declared_mxn" / $"mtu_observed_p95_mxn"
    ).otherwise(lit(null))
  )
  // -- 8. Convenience booleans
  .withColumn("is_scam", coalesce($"confirmed_scam", lit(false)))
  .withColumn("policy_acted", $"action_taken".isNotNull && $"action_taken" =!= "none")
  .withColumn("was_blocked", $"action_taken" === "delay")
  .withColumn("was_warned", $"action_taken" === "scam_alert")

println(s"master: ${master.count()} rows, ${master.columns.length} cols")
master.printSchema()

In [ ]:
// Sanity: distribution of key flags
println("=== Scam flag ===")
master.groupBy("is_scam").count().show()

println("=== Policy action distribution ===")
master.groupBy("action_taken").count().orderBy($"count".desc).show()

println("=== Holdout flag (within policy events) ===")
master.filter($"rule_id".isNotNull).groupBy("policy_holdout_flag").count().show()

println("=== Completed flag ===")
master.groupBy("completed_flag").count().show()

println("=== mtu_ratio stats ===")
master.select(
  mean("mtu_ratio").as("avg"),
  expr("percentile_approx(mtu_ratio, 0.5)").as("p50"),
  expr("percentile_approx(mtu_ratio, 0.95)").as("p95"),
  max("mtu_ratio").as("max")
).show()

In [ ]:
// Check how many txn_ids from policy_events and scam_reports exist in the transactions sample
val txnIds = transactions.select("txn_id").distinct()
val peIds  = policyEvents.select("txn_id").distinct()
val srIds  = scamReports.select("txn_id").distinct()

val peOverlap = txnIds.join(peIds, Seq("txn_id"), "inner").count()
val srOverlap = txnIds.join(srIds, Seq("txn_id"), "inner").count()
val peSrOverlap = peIds.join(srIds, Seq("txn_id"), "inner").count()

println(s"txn_ids in transactions sample:   ${txnIds.count()}")
println(s"txn_ids in policy_events sample:  ${peIds.count()}")
println(s"txn_ids in scam_reports sample:   ${srIds.count()}")
println(s"")
println(s"Overlap txn ∩ policy_events:  $peOverlap")
println(s"Overlap txn ∩ scam_reports:   $srOverlap")
println(s"Overlap policy_events ∩ scam:  $peSrOverlap")

### Point 1 — Current Policy Diagnosis (rule by rule)
Analyze each rule (P-01 to P-05): volume triggered, friction generated, fraud prevented. Use holdout group for counterfactual.

In [ ]:
// Join policy_events with scam_reports to see which triggered txns were scams
val policyWithScam = policyEvents
  .join(scamReports.select("txn_id", "confirmed_scam", "loss_amount_mxn"), Seq("txn_id"), "left")
  .withColumn("is_scam", coalesce($"confirmed_scam", lit(false)))

// Also join customer info for richer context
val policyFull = policyWithScam
  .join(transactions.select("txn_id", "customer_id", "amount_mxn", "channel", 
    "counterparty_first_seen_flag", "device_new_flag", "completed_flag"), Seq("txn_id"), "left")
  .join(customerMtu, Seq("customer_id"), "left")

println(s"Policy events with scam join: ${policyWithScam.count()} rows")
println(s"  - matched scam reports: ${policyWithScam.filter($"confirmed_scam" === true).count()}")
println(s"  - matched transactions: ${policyFull.filter($"amount_mxn".isNotNull).count()}")

In [ ]:
// Rule-by-rule breakdown: volume, actions, friction, scam catch rate
val ruleAnalysis = policyWithScam
  .groupBy("rule_id", "rule_description")
  .agg(
    count("*").as("total_triggered"),
    // Action breakdown
    sum(when($"action_taken" === "delay", 1).otherwise(0)).as("delays"),
    sum(when($"action_taken" === "scam_alert", 1).otherwise(0)).as("warnings"),
    sum(when($"action_taken" === "none", 1).otherwise(0)).as("holdout_none"),
    // Friction metrics
    sum("minutes_blocked").as("total_minutes_blocked"),
    avg("minutes_blocked").as("avg_minutes_blocked"),
    sum(when($"ops_contact_flag" === true, 1).otherwise(0)).as("ops_contacts"),
    // Warning pass-through
    sum(when($"customer_proceeded" === true, 1).otherwise(0)).as("proceeded_past_warning"),
    // Bypass
    sum(when($"bypass_requested" === true, 1).otherwise(0)).as("bypass_requests"),
    sum(when($"bypass_granted" === true, 1).otherwise(0)).as("bypass_granted"),
    // Scam detection
    sum(when($"is_scam", 1).otherwise(0)).as("scams_caught"),
    sum(when($"is_scam", $"loss_amount_mxn").otherwise(0.0)).as("scam_loss_caught"),
    // MTU breach
    sum(when($"mtu_breach_flag" === true, 1).otherwise(0)).as("mtu_breaches"),
    // Holdout scam rate (key counterfactual)
    sum(when($"policy_holdout_flag" === true && $"is_scam", 1).otherwise(0)).as("holdout_scams"),
    sum(when($"policy_holdout_flag" === true, 1).otherwise(0)).as("holdout_total")
  )
  .orderBy("rule_id")

display(ruleAnalysis)

In [ ]:
// Overall policy effectiveness summary
println("=== Overall Policy Summary ===")
val totalPolicyTxns = policyWithScam.count()
val totalDelays = policyWithScam.filter($"action_taken" === "delay").count()
val totalWarnings = policyWithScam.filter($"action_taken" === "scam_alert").count()
val totalHoldout = policyWithScam.filter($"policy_holdout_flag" === true).count()
val totalScamsCaught = policyWithScam.filter($"is_scam").count()
val totalMinBlocked = policyWithScam.agg(sum("minutes_blocked")).head().getAs[Long](0)
val totalOpsContacts = policyWithScam.filter($"ops_contact_flag" === true).count()

println(s"Total policy-triggered txns:  $totalPolicyTxns")
println(s"  Delays (blocked 12h):       $totalDelays")
println(s"  Scam alerts (warning):      $totalWarnings")
println(s"  Holdout (no action):        $totalHoldout")
println(s"  Scams identified:           $totalScamsCaught")
println(s"  Total minutes blocked:      $totalMinBlocked")
println(s"  Ops contacts generated:     $totalOpsContacts")

// False positive proxy: delays on non-scam transactions
val falseDelays = policyWithScam.filter($"action_taken" === "delay" && !$"is_scam").count()
val falseWarnings = policyWithScam.filter($"action_taken" === "scam_alert" && !$"is_scam").count()
println(s"\n  False delays (legit blocked):  $falseDelays / $totalDelays")
println(s"  False warnings (legit warned): $falseWarnings / $totalWarnings")

// Holdout counterfactual
val holdoutScams = policyWithScam.filter($"policy_holdout_flag" === true && $"is_scam").count()
println(s"\n  Holdout scam rate: $holdoutScams / $totalHoldout (counterfactual)")

### Point 2 — Feature Exploration for Fraud Model
Analyze scam vs legit transactions on all available features. Since samples are independent, we build a transaction-level enriched view for modeling.

In [ ]:
// Build enriched transaction-level view joining all available info
val txnEnriched = transactions
  .join(customerMtu, Seq("customer_id"), "left")
  .join(scamReports.select("txn_id", "confirmed_scam", "loss_amount_mxn"), Seq("txn_id"), "left")
  .join(policyEvents.select("txn_id", "rule_id", "action_taken", "policy_holdout_flag"), Seq("txn_id"), "left")
  .withColumn("is_scam", coalesce($"confirmed_scam", lit(false)))
  .withColumn("mtu_ratio",
    when($"mtu_declared_mxn" > 0,
      ($"mtd_volume_before_mxn" + $"amount_mxn") / $"mtu_declared_mxn"
    ).otherwise(lit(null))
  )
  .withColumn("geo_mismatch", $"geo_state" =!= $"home_state")
  .withColumn("ticket_ratio",
    when($"avg_ticket_90d_mxn" > 0, $"amount_mxn" / $"avg_ticket_90d_mxn").otherwise(lit(null))
  )
  .withColumn("mtu_gap_ratio",
    when($"mtu_observed_p95_mxn" > 0, $"mtu_declared_mxn" / $"mtu_observed_p95_mxn").otherwise(lit(null))
  )

println(s"txnEnriched: ${txnEnriched.count()} rows")
println(s"  scams: ${txnEnriched.filter($"is_scam").count()}")
println(s"  with policy event: ${txnEnriched.filter($"rule_id".isNotNull).count()}")

In [ ]:
// Also analyze from the scam_reports table side:
// Join scam reports back to transactions to get features of scam txns
val scamTxns = scamReports
  .filter($"confirmed_scam" === true)
  .join(transactions, Seq("txn_id"), "left")
  .join(customerMtu, Seq("customer_id"), "left")
  .withColumn("mtu_ratio",
    when($"mtu_declared_mxn" > 0,
      ($"mtd_volume_before_mxn" + $"amount_mxn") / $"mtu_declared_mxn"
    ).otherwise(lit(null))
  )
  .withColumn("geo_mismatch", $"geo_state" =!= $"home_state")
  .withColumn("ticket_ratio",
    when($"avg_ticket_90d_mxn" > 0, $"amount_mxn" / $"avg_ticket_90d_mxn").otherwise(lit(null))
  )

val confirmedScams = scamReports.filter($"confirmed_scam" === true).count()
val matchedToTxn = scamTxns.filter($"amount_mxn".isNotNull).count()
println(s"Confirmed scams in scam_reports: $confirmedScams")
println(s"  matched to transactions sample: $matchedToTxn")

// Analyze scam_reports on its own
println("\n=== Scam reports summary ===")
scamReports.groupBy("confirmed_scam").agg(
  count("*").as("count"),
  avg("loss_amount_mxn").as("avg_loss"),
  sum("loss_amount_mxn").as("total_loss")
).show()

println("=== Scam report channels ===")
scamReports.filter($"confirmed_scam" === true)
  .groupBy("report_channel")
  .agg(count("*").as("count"), avg("loss_amount_mxn").as("avg_loss"))
  .orderBy($"count".desc).show()

In [ ]:
// For the scams that DO match transactions, profile the features
if (matchedToTxn > 0) {
  println("=== Scam transactions - feature profile ===")
  scamTxns.filter($"amount_mxn".isNotNull).select(
    "amount_mxn", "channel", "counterparty_first_seen_flag", "device_new_flag",
    "geo_state", "home_state", "hour_of_day", "is_weekend", "mtu_ratio", "geo_mismatch",
    "ticket_ratio", "risk_segment", "prior_scam_report_flag", "tenure_months", "income_band"
  ).show(false)
}

// Compare average feature values: scam vs legit in the transactions sample
println("=== Feature comparison (from txnEnriched) ===")
txnEnriched.groupBy("is_scam").agg(
  count("*").as("n"),
  avg("amount_mxn").as("avg_amount"),
  avg("mtu_ratio").as("avg_mtu_ratio"),
  avg("ticket_ratio").as("avg_ticket_ratio"),
  avg(when($"counterparty_first_seen_flag", 1).otherwise(0).cast("double")).as("pct_new_counterparty"),
  avg(when($"device_new_flag", 1).otherwise(0).cast("double")).as("pct_new_device"),
  avg(when($"geo_mismatch", 1).otherwise(0).cast("double")).as("pct_geo_mismatch"),
  avg("hour_of_day").as("avg_hour"),
  avg(when($"is_weekend", 1).otherwise(0).cast("double")).as("pct_weekend"),
  avg(when($"prior_scam_report_flag", 1).otherwise(0).cast("double")).as("pct_prior_scam")
).show(false)

### Point 4 — Emerging Scam Pattern Detection
Look for behavioral patterns in the scam data that the current policy misses. Cross-reference scam reports with policy events to find the gap.

In [ ]:
// How many confirmed scams were caught by the policy vs slipped through entirely?
val scamConfirmed = scamReports.filter($"confirmed_scam" === true)
val scamWithPolicy = scamConfirmed
  .join(policyEvents.select("txn_id", "rule_id", "action_taken", "policy_holdout_flag"), Seq("txn_id"), "left")

val caughtByPolicy = scamWithPolicy.filter($"rule_id".isNotNull).count()
val missedByPolicy = scamWithPolicy.filter($"rule_id".isNull).count()
val totalConfirmed = scamConfirmed.count()

println(s"Confirmed scams: $totalConfirmed")
println(s"  Caught by policy (had a rule fire): $caughtByPolicy")
println(s"  Missed by policy entirely:          $missedByPolicy")
println(s"  Policy catch rate:                  ${caughtByPolicy.toDouble / totalConfirmed}")

// Which rules caught scams?
println("\n=== Rules that fired on scam txns ===")
scamWithPolicy.filter($"rule_id".isNotNull)
  .groupBy("rule_id", "action_taken")
  .agg(count("*").as("count"), sum("loss_amount_mxn").as("total_loss"))
  .orderBy($"count".desc)
  .show()

In [ ]:
// Missed scams: analyze from the scam_reports side
val missedScams = scamWithPolicy.filter($"rule_id".isNull)
println(s"Missed scams: ${missedScams.count()} / $totalConfirmed")
println(s"Missed scam loss: ${missedScams.agg(sum("loss_amount_mxn")).head().getDouble(0)} MXN")

// Report channel of missed scams
println("\n=== Report channel of missed scams ===")
missedScams.groupBy("report_channel")
  .agg(count("*").as("count"), avg("loss_amount_mxn").as("avg_loss"), sum("loss_amount_mxn").as("total_loss"))
  .orderBy($"count".desc).show()

In [ ]:
// Temporal analysis: are scams clustering in recent weeks?
val scamTemporal = scamConfirmed
  .withColumn("report_week", weekofyear($"reported_ts"))
  .withColumn("report_date", to_date($"reported_ts"))

println("=== Scam reports by week ===")
scamTemporal.groupBy("report_week")
  .agg(
    count("*").as("reports"),
    sum("loss_amount_mxn").as("total_loss"),
    avg("loss_amount_mxn").as("avg_loss")
  )
  .orderBy("report_week").show(30)

println("=== Time range ===")
scamTemporal.select(min("reported_ts"), max("reported_ts")).show()

In [ ]:
// Loss amount distribution for confirmed scams
println("=== Loss amount stats ===")
scamConfirmed.select(
  count("*").as("n"),
  avg("loss_amount_mxn").as("avg"),
  expr("percentile_approx(loss_amount_mxn, 0.25)").as("p25"),
  expr("percentile_approx(loss_amount_mxn, 0.50)").as("p50"),
  expr("percentile_approx(loss_amount_mxn, 0.75)").as("p75"),
  expr("percentile_approx(loss_amount_mxn, 0.95)").as("p95"),
  max("loss_amount_mxn").as("max")
).show()

// Loss buckets
println("=== Loss buckets ===")
scamConfirmed
  .withColumn("loss_bucket", 
    when($"loss_amount_mxn" < 500, "a_0-500")
    .when($"loss_amount_mxn" < 1000, "b_500-1K")
    .when($"loss_amount_mxn" < 2000, "c_1K-2K")
    .when($"loss_amount_mxn" < 5000, "d_2K-5K")
    .when($"loss_amount_mxn" < 10000, "e_5K-10K")
    .otherwise("f_10K+")
  )
  .groupBy("loss_bucket").agg(
    count("*").as("count"),
    sum("loss_amount_mxn").as("total_loss")
  ).orderBy("loss_bucket").show()

In [ ]:
// Analyze scams the policy caught: what was the outcome?
println("=== Scams caught by policy - what happened? ===")
scamWithPolicy.filter($"rule_id".isNotNull)
  .groupBy("action_taken")
  .agg(
    count("*").as("count"),
    sum("loss_amount_mxn").as("total_loss"),
    avg("loss_amount_mxn").as("avg_loss")
  ).show()

// Holdout scams: these got no intervention despite matching a rule
println("=== Holdout scams (rule matched but no action) ===")
scamWithPolicy
  .filter($"rule_id".isNotNull && $"policy_holdout_flag" === true)
  .select("txn_id", "rule_id", "action_taken", "loss_amount_mxn")
  .show(20)

// Key insight: what fraction of scam losses come from policy-missed txns?
val totalScamLoss = scamConfirmed.agg(sum("loss_amount_mxn")).head().getDouble(0)
val missedScamLoss = missedScams.agg(sum("loss_amount_mxn")).head().getDouble(0)
val caughtScamLoss = totalScamLoss - missedScamLoss
println(s"\nTotal confirmed scam losses:  $$totalScamLoss MXN")
println(s"  Losses from policy-missed:   $$missedScamLoss MXN (${missedScamLoss / totalScamLoss})")
println(s"  Losses where policy fired:   $$caughtScamLoss MXN (${caughtScamLoss / totalScamLoss})")

In [ ]:
// Look for an emerging pattern: is there a recent surge in avg scam size?
// Week 15 has highest avg_loss ($3,100). Let's drill deeper.
val scamWithTxnFeatures = scamConfirmed
  .join(transactions, Seq("txn_id"), "left")
  .join(customerMtu, Seq("customer_id"), "left")
  .withColumn("report_week", weekofyear($"reported_ts"))

// For scams that matched transactions, analyze features by week
println("=== Scam txns matched to transactions, by week ===")
scamWithTxnFeatures.filter($"amount_mxn".isNotNull)
  .groupBy("report_week")
  .agg(
    count("*").as("n"),
    avg("amount_mxn").as("avg_amount"),
    avg(when($"counterparty_first_seen_flag", 1).otherwise(0).cast("double")).as("pct_new_cp"),
    avg(when($"device_new_flag", 1).otherwise(0).cast("double")).as("pct_new_device"),
    avg("hour_of_day").as("avg_hour")
  ).orderBy("report_week").show()

// Even without full txn match: analyze loss distribution in the last vs earlier weeks
println("=== Recent (week >= 14) vs earlier scams ===")
val withPeriod = scamConfirmed
  .withColumn("report_week", weekofyear($"reported_ts"))
  .withColumn("period", when($"report_week" >= 14, "recent").otherwise("earlier"))

withPeriod.groupBy("period").agg(
  count("*").as("n"),
  avg("loss_amount_mxn").as("avg_loss"),
  sum("loss_amount_mxn").as("total_loss"),
  expr("percentile_approx(loss_amount_mxn, 0.75)").as("p75_loss"),
  expr("percentile_approx(loss_amount_mxn, 0.95)").as("p95_loss")
).show()

In [ ]:
// Look at the high-value tail: scams > $5K MXN
val highValueScams = scamConfirmed.filter($"loss_amount_mxn" >= 5000)
val hvWithPolicy = highValueScams
  .join(policyEvents.select("txn_id", "rule_id", "action_taken"), Seq("txn_id"), "left")

println(s"High-value scams (>= 5K MXN): ${highValueScams.count()}")
println(s"  Caught by policy: ${hvWithPolicy.filter($"rule_id".isNotNull).count()}")
println(s"  Missed by policy: ${hvWithPolicy.filter($"rule_id".isNull).count()}")
println(s"  Total loss: ${highValueScams.agg(sum("loss_amount_mxn")).head().getDouble(0)} MXN")

// Channel distribution for high-value missed scams
println("\n=== High-value missed scams by report channel ===")
hvWithPolicy.filter($"rule_id".isNull)
  .groupBy("report_channel")
  .agg(count("*").as("count"), sum("loss_amount_mxn").as("total_loss"))
  .orderBy($"total_loss".desc).show()

// Week distribution for high-value scams
println("=== High-value scams by week ===")
highValueScams
  .withColumn("report_week", weekofyear($"reported_ts"))
  .groupBy("report_week")
  .agg(count("*").as("count"), sum("loss_amount_mxn").as("total_loss"), avg("loss_amount_mxn").as("avg_loss"))
  .orderBy("report_week").show()

In [ ]:
// Friction cost analysis: what is the policy costing in terms of customer experience?
println("=== FRICTION COST SUMMARY ===")

// From policy_events: legitimate transactions affected
val legitPolicyTxns = policyWithScam.filter(!$"is_scam")
val totalLegitDelayed = legitPolicyTxns.filter($"action_taken" === "delay").count()
val totalLegitWarned = legitPolicyTxns.filter($"action_taken" === "scam_alert").count()
val totalLegitHours = legitPolicyTxns.agg(sum("minutes_blocked")).head().getAs[Long](0) / 60.0
val totalLegitOps = legitPolicyTxns.filter($"ops_contact_flag" === true).count()
val totalBypassReq = legitPolicyTxns.filter($"bypass_requested" === true).count()
val totalBypassGranted = legitPolicyTxns.filter($"bypass_granted" === true).count()
val totalProceeded = legitPolicyTxns.filter($"customer_proceeded" === true).count()

println(s"Legitimate customers impacted:")
println(s"  Delayed (12h block):     $totalLegitDelayed")
println(s"  Warned (scam alert):     $totalLegitWarned")
println(s"  Total hours blocked:     ${f"$totalLegitHours%.0f"} hours")
println(s"  Ops contacts:            $totalLegitOps")
println(s"  Bypass requests:         $totalBypassReq")
println(s"  Bypass granted:          $totalBypassGranted")
println(s"  Proceeded past warning:  $totalProceeded")

val caughtScams = totalScamsCaught
val caughtLoss = totalScamLoss - missedScamLoss
println(s"\nScam prevention achieved:")
println(s"  Scams caught: $caughtScams out of $totalConfirmed confirmed (${caughtScams.toDouble / totalConfirmed})")
println(s"  Losses caught: $caughtLoss out of $totalScamLoss (${caughtLoss / totalScamLoss})")
println(s"\nFriction-to-fraud ratio:")
if (caughtScams > 0) {
  println(s"  Legitimate delays per scam caught: ${totalLegitDelayed.toDouble / caughtScams}")
  println(s"  Hours blocked per dollar saved:    ${totalLegitHours / caughtLoss}")
} else {
  println("  No scams caught - ratio undefined")
}

### Point 2/3 — Fraud Scoring Model & Policy Proposal
Train a classifier on the master DataFrame. Use Python (sklearn/XGBoost) for better imbalanced-classification tools. Map the score to the three policy actions.

In [ ]:
// Select features and label for ML, write to temp view for Python
// FIX Error 6: carry txn_ts + week for temporal split
// FIX Error 7: DO NOT na.fill(0.0) — imputation done in Python with median + missing indicators
// Also carry mtu_ratio raw + channel + mtu_declared for baseline policies
val modelData = master.select(
  $"txn_id",
  $"txn_ts",
  weekofyear($"txn_ts").as("txn_week"),
  $"amount_mxn".cast("double"),
  $"mtu_ratio",
  $"ticket_ratio",
  $"mtu_gap_ratio",
  when($"counterparty_first_seen_flag", 1.0).otherwise(0.0).as("new_counterparty"),
  when($"device_new_flag", 1.0).otherwise(0.0).as("new_device"),
  when($"geo_mismatch", 1.0).otherwise(0.0).as("geo_mismatch_flag"),
  $"hour_of_day".cast("double"),
  when($"is_weekend", 1.0).otherwise(0.0).as("is_weekend_flag"),
  when($"prior_scam_report_flag", 1.0).otherwise(0.0).as("prior_scam"),
  $"tenure_months".cast("double"),
  $"avg_ticket_90d_mxn",
  $"mtu_observed_p95_mxn",
  $"mtu_declared_mxn",
  $"mtd_volume_before_mxn".cast("double"),
  // Encode channel as dummies
  $"channel",
  when($"channel" === "spei_out", 1.0).otherwise(0.0).as("ch_spei_out"),
  when($"channel" === "card_online", 1.0).otherwise(0.0).as("ch_card_online"),
  when($"channel" === "cash_out", 1.0).otherwise(0.0).as("ch_cash_out"),
  when($"channel" === "p2p_nu", 1.0).otherwise(0.0).as("ch_p2p_nu"),
  when($"channel" === "card_present", 1.0).otherwise(0.0).as("ch_card_present"),
  // Encode risk_segment
  when($"risk_segment" === "high", 1.0).otherwise(0.0).as("risk_high"),
  when($"risk_segment" === "medium", 1.0).otherwise(0.0).as("risk_medium"),
  // Label
  when($"is_scam", 1.0).otherwise(0.0).as("label"),
  when($"completed_flag", 1.0).otherwise(0.0).as("completed"),
  $"loss_amount_mxn"
)  // NO na.fill — Python handles imputation

modelData.createOrReplaceTempView("model_data")
println(s"Model data: ${modelData.count()} rows, ${modelData.columns.length} cols")
println(s"Label distribution:")
modelData.groupBy("label").count().show()

// Show null counts in key numeric columns
println("=== Null counts in key features ===")
val nullCols = Seq("mtu_ratio", "ticket_ratio", "mtu_gap_ratio", "tenure_months", "prior_scam", "hour_of_day")
nullCols.foreach { c =>
  val nullCnt = modelData.filter(col(c).isNull).count()
  println(s"  $c: $nullCnt nulls")
}

In [ ]:
%python
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, classification_report
)
import warnings
warnings.filterwarnings('ignore')

# ---- Load ----
pdf = spark.table("model_data").toPandas()

# ---- Feature columns (model inputs) ----
numeric_cols = [
    'amount_mxn', 'mtu_ratio', 'ticket_ratio', 'mtu_gap_ratio',
    'hour_of_day', 'tenure_months'
]
binary_cols = [
    'new_counterparty', 'new_device', 'geo_mismatch_flag',
    'is_weekend_flag', 'prior_scam',
    'ch_spei_out', 'ch_card_online', 'ch_cash_out', 'ch_p2p_nu', 'ch_card_present',
    'risk_high', 'risk_medium'
]

# ---- FIX Error 7: median imputation + missing indicators ----
for c in numeric_cols:
    mask = pdf[c].isna()
    if mask.any():
        med = pdf.loc[~mask, c].median()
        pdf[f'{c}_missing'] = mask.astype(float)
        pdf[c] = pdf[c].fillna(med)
        print(f"  {c}: {mask.sum()} nulls → imputed with median {med:.4f}, added {c}_missing")

# Binary cols: fill NaN with 0 (absence of flag)
for c in binary_cols:
    pdf[c] = pdf[c].fillna(0.0)

# Build final feature list (numeric + their _missing indicators + binary)
missing_cols = [c for c in pdf.columns if c.endswith('_missing')]
feature_cols = numeric_cols + missing_cols + binary_cols
print(f"\nTotal features: {len(feature_cols)}")

# ---- FIX Error 6: temporal split ----
# Train: weeks <= 23  |  Test: weeks >= 24 (regime change)
train_mask = pdf['txn_week'] <= 23
test_mask  = pdf['txn_week'] >= 24

X_train = pdf.loc[train_mask, feature_cols]
y_train = pdf.loc[train_mask, 'label']
X_test  = pdf.loc[test_mask, feature_cols]
y_test  = pdf.loc[test_mask, 'label']
pdf_test = pdf.loc[test_mask].copy()
pdf_train = pdf.loc[train_mask].copy()

print(f"\n=== Temporal Split ===")
print(f"Train (wk<=23): {len(X_train):,} rows, {y_train.sum():.0f} scams ({y_train.mean():.6f})")
print(f"Test  (wk>=24): {len(X_test):,} rows, {y_test.sum():.0f} scams ({y_test.mean():.6f})")

# ---- Train GBM with class weighting ----
scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"Train imbalance ratio: {scale_pos:.0f}:1")

sample_w = np.where(y_train == 1, scale_pos, 1.0)

model = GradientBoostingClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    min_samples_leaf=100,
    random_state=42
)
model.fit(X_train, y_train, sample_weight=sample_w)

y_prob_test  = model.predict_proba(X_test)[:, 1]
y_prob_train = model.predict_proba(X_train)[:, 1]

print(f"\nTrain AUC-ROC: {roc_auc_score(y_train, y_prob_train):.4f}")
print(f"Test  AUC-ROC: {roc_auc_score(y_test, y_prob_test):.4f}")
print(f"Test  AUC-PR:  {average_precision_score(y_test, y_prob_test):.4f}")

In [ ]:
%python
# ---- Feature importance ----
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Feature Importance ===")
for _, row in importances.iterrows():
    if row['importance'] >= 0.005:
        print(f"  {row['feature']:25s} {row['importance']:.4f}")

# ---- FIX Error 8: leakage check on snapshot features ----
# If prior_scam_report_flag, avg_ticket_90d_mxn, mtu_observed_p95_mxn were
# computed at end-of-period, early-week scams would show suspiciously high rates.
print("\n=== Error 8: Temporal Leakage Check ===")
print("Scam rate among prior_scam_report_flag=true, by week bucket:")
leak_check = pdf_train.copy()
leak_check['week_bucket'] = pd.cut(leak_check['txn_week'], bins=[0, 12, 18, 23], labels=['wk1-12', 'wk13-18', 'wk19-23'])
for wb in ['wk1-12', 'wk13-18', 'wk19-23']:
    sub = leak_check[leak_check['week_bucket'] == wb]
    flagged = sub[sub['prior_scam'] == 1]
    if len(flagged) > 0:
        rate = flagged['label'].mean()
        print(f"  {wb}: {len(flagged):,} txns with prior_scam=1, scam rate = {rate:.4f}")
    else:
        print(f"  {wb}: no txns with prior_scam=1")

# Also train a model WITHOUT the 3 potentially leaky features
leaky_cols = ['prior_scam', 'tenure_months']  # avg_ticket_90d, mtu_observed_p95 not directly used as features
feature_cols_safe = [c for c in feature_cols if c not in leaky_cols and not c.startswith('tenure_months')]

print(f"\n=== Model WITHOUT potentially leaky features ({len(feature_cols_safe)} features) ===")
model_safe = GradientBoostingClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, min_samples_leaf=100, random_state=42
)
sample_w_train = np.where(y_train == 1, scale_pos, 1.0)
model_safe.fit(X_train[feature_cols_safe], y_train, sample_weight=sample_w_train)
y_prob_test_safe = model_safe.predict_proba(X_test[feature_cols_safe])[:, 1]
print(f"Test AUC-ROC (safe): {roc_auc_score(y_test, y_prob_test_safe):.4f}")
print(f"Test AUC-PR  (safe): {average_precision_score(y_test, y_prob_test_safe):.4f}")
print(f"\nFull model AUC-ROC: {roc_auc_score(y_test, y_prob_test):.4f} (for comparison)")
print("If safe model is close to full model, leakage is not a concern.")

In [ ]:
%python
# ===============================================================
# FIX Errors 1,2,3,5: Policy optimization on EXPECTED LOSS
# - Thresholds on expected_loss = fraud_score * amount_mxn (Error 3)
# - Net value objective with business params (Error 1)
# - Extended grid (Error 2)
# - Effectiveness coefficients on prevented loss (Error 5)
# ===============================================================

# ---- Business parameters (from policy_events observed data) ----
V_HORA  = 10.0    # MXN cost per hour of blocking a legit customer
C_WARN  = 2.0     # MXN cost per warning (attention + alert fatigue)
C_OPS   = 50.0    # MXN cost per ops contact
E_DELAY = 0.758   # effectiveness of delay (1 - bypass_granted_rate)
E_WARN  = 0.215   # effectiveness of warning (1 - proceed_rate)
HRS     = 9.0     # avg hours blocked (543 min observed)
P_OPS   = 0.34    # prob of ops contact given delay
BASE_RATE = y_test.mean()   # 0.00786 en el test temporal

# ---- FIX Error 3: decide on expected loss, not raw score ----
pdf_test = pdf_test.copy()
pdf_test['fraud_score'] = y_prob_test
pdf_test['expected_loss'] = y_prob_test * pdf_test['amount_mxn']

is_scam = pdf_test['label'] == 1
loss = pdf_test['loss_amount_mxn'].fillna(0)
amt  = pdf_test['amount_mxn']
n_test = len(pdf_test)

# ---- FIX Error 2: grid in MXN of expected loss, wide range ----
warn_grid  = np.concatenate([np.arange(5, 100, 5), np.arange(100, 1001, 25)])
delay_grid = np.concatenate([np.arange(50, 500, 25), np.arange(500, 5001, 100)])

results = []
for warn_t in warn_grid:
    for delay_t in delay_grid:
        if delay_t <= warn_t:
            continue
        
        actions = np.where(pdf_test['expected_loss'] >= delay_t, 'delay',
                  np.where(pdf_test['expected_loss'] >= warn_t, 'scam_alert', 'let_through'))
        
        scam_delayed = (actions == 'delay') & is_scam
        scam_warned  = (actions == 'scam_alert') & is_scam
        scam_through = (actions == 'let_through') & is_scam
        legit_delayed = ((actions == 'delay') & ~is_scam).sum()
        legit_warned  = ((actions == 'scam_alert') & ~is_scam).sum()
        
        n_delayed = (actions == 'delay').sum()
        n_warned  = (actions == 'scam_alert').sum()
        
        # Precision per bucket
        prec_delay = scam_delayed.sum() / n_delayed if n_delayed > 0 else 0
        prec_warn  = scam_warned.sum() / n_warned if n_warned > 0 else 0
        
        # ---- Guardrail: discard if any active bucket is below base rate ----
        if n_delayed > 0 and prec_delay < BASE_RATE:
            continue
        if n_warned > 0 and prec_warn < BASE_RATE:
            continue
        # ---- Guardrail: warn bucket <= 10% of transactions ----
        if n_warned > 0.10 * n_test:
            continue
        
        # ---- FIX Error 5: apply effectiveness coefficients ----
        loss_exposure_delayed = loss[scam_delayed].sum()
        loss_exposure_warned  = loss[scam_warned].sum()
        loss_through_val      = loss[scam_through].sum()
        
        loss_prevented_delay = E_DELAY * loss_exposure_delayed
        loss_prevented_warn  = E_WARN  * loss_exposure_warned
        
        # ---- FIX Error 1: net value objective ----
        beneficio  = loss_prevented_delay + loss_prevented_warn
        costo      = legit_delayed * (HRS * V_HORA + P_OPS * C_OPS) + legit_warned * C_WARN
        valor_neto = beneficio - costo
        
        # Efficiency as reporting metric only
        total_legit_hrs = legit_delayed * HRS
        efficiency = beneficio / total_legit_hrs if total_legit_hrs > 0 else 0
        
        results.append({
            'warn_t': warn_t, 'delay_t': delay_t,
            'scams_delayed': scam_delayed.sum(), 'scams_warned': scam_warned.sum(),
            'scams_through': scam_through.sum(),
            'loss_exposure_delayed': loss_exposure_delayed,
            'loss_exposure_warned': loss_exposure_warned,
            'loss_prevented_est': beneficio,
            'loss_through': loss_through_val,
            'legit_delayed': legit_delayed, 'legit_warned': legit_warned,
            'prec_delay': prec_delay, 'prec_warn': prec_warn,
            'costo': costo, 'valor_neto': valor_neto,
            'efficiency': efficiency,
            'pct_warned': n_warned / n_test,
        })

results_df = pd.DataFrame(results)
print(f"Valid configurations after guardrails: {len(results_df)}")

# ---- Sort by valor_neto (net value), NOT efficiency ----
print("\n=== Top 10 Policies by Net Value (MXN) ===")
top = results_df.nlargest(10, 'valor_neto')
for _, r in top.iterrows():
    print(f"  warn={r['warn_t']:.0f} delay={r['delay_t']:.0f} MXN | "
          f"scam: {r['scams_delayed']:.0f}d/{r['scams_warned']:.0f}w/{r['scams_through']:.0f}t | "
          f"legit: {r['legit_delayed']:.0f}d/{r['legit_warned']:.0f}w | "
          f"prevented(est): ${r['loss_prevented_est']:,.0f} | "
          f"cost: ${r['costo']:,.0f} | "
          f"NET: ${r['valor_neto']:,.0f} | "
          f"prec_d:{r['prec_delay']:.3f} prec_w:{r['prec_warn']:.3f}")

# Check optimal is not at grid edge
best = results_df.nlargest(1, 'valor_neto').iloc[0]
print(f"\n=== Grid Edge Check ===")
print(f"Best warn_t={best['warn_t']:.0f}, grid range [{warn_grid.min()}, {warn_grid.max()}]")
print(f"Best delay_t={best['delay_t']:.0f}, grid range [{delay_grid.min()}, {delay_grid.max()}]")
if best['warn_t'] in [warn_grid.min(), warn_grid.max()]:
    print("WARNING: warn threshold at grid edge!")
if best['delay_t'] in [delay_grid.min(), delay_grid.max()]:
    print("WARNING: delay threshold at grid edge!")

In [ ]:
%python
# ===============================================================
# EXTRA: Calibración del score y re-optimización de umbrales
# El peso de clase (456:1) infla las probabilidades predichas.
# Corrección analítica de odds -> pérdida esperada en MXN reales.
# No toca nada anterior: todo se guarda con sufijo _cal
# ===============================================================

# ---- 1. Calibración por corrección de odds ----
w = scale_pos
odds_raw = y_prob_test / (1 - y_prob_test + 1e-12)
p_cal = (odds_raw / w) / (1 + odds_raw / w)

print("=== Verificación de calibración ===")
print(f"  Media del score crudo:       {y_prob_test.mean():.5f}")
print(f"  Media del score calibrado:   {p_cal.mean():.5f}")
print(f"  Prevalencia real (test):     {y_test.mean():.5f}")
print("  (el calibrado debe acercarse a la prevalencia; el crudo no)")

pdf_test['fraud_score_cal']   = p_cal
pdf_test['expected_loss_cal'] = p_cal * pdf_test['amount_mxn']
el = pdf_test['expected_loss_cal'].values

print(f"\n  Pérdida esperada calibrada — p50: ${np.percentile(el,50):,.2f} | "
      f"p95: ${np.percentile(el,95):,.2f} | max: ${el.max():,.2f}")

# ---- 2. Umbrales teóricos, derivados de la economía (no de los datos) ----
UMBRAL_DELAY_TEO = (HRS*V_HORA + P_OPS*C_OPS) / (E_DELAY - E_WARN)
UMBRAL_WARN_TEO  = C_WARN / E_WARN
print(f"\n=== Umbrales teóricos (primeros principios) ===")
print(f"  Advertir si pérdida esperada > ${UMBRAL_WARN_TEO:,.2f}")
print(f"  Demorar  si pérdida esperada > ${UMBRAL_DELAY_TEO:,.2f}")

# ---- 3. Re-optimización: la rejilla cambia de escala ----
warn_grid_cal  = np.arange(2, 300, 2)
delay_grid_cal = np.arange(20, 2000, 10)

is_s = is_scam.values
results_cal = []
for warn_t in warn_grid_cal:
    for delay_t in delay_grid_cal:
        if delay_t <= warn_t:
            continue
        actions = np.where(el >= delay_t, 'delay',
                  np.where(el >= warn_t, 'scam_alert', 'let_through'))
        sd = (actions == 'delay') & is_s
        sw = (actions == 'scam_alert') & is_s
        nd = (actions == 'delay').sum()
        nw = (actions == 'scam_alert').sum()
        ld = ((actions == 'delay') & ~is_s).sum()
        lw = ((actions == 'scam_alert') & ~is_s).sum()

        prec_d = sd.sum()/nd if nd > 0 else 0
        prec_w = sw.sum()/nw if nw > 0 else 0

        # Mismos guardrails que la versión sin calibrar
        if nd > 0 and prec_d < BASE_RATE:  continue
        if nw > 0 and prec_w < BASE_RATE:  continue
        if nw > 0.10 * n_test:             continue

        prevented = E_DELAY*loss[sd].sum() + E_WARN*loss[sw].sum()
        costo_v   = ld*(HRS*V_HORA + P_OPS*C_OPS) + lw*C_WARN
        results_cal.append({
            'warn_t': warn_t, 'delay_t': delay_t,
            'valor_neto': prevented - costo_v,
            'loss_prevented_est': prevented, 'costo': costo_v,
            'legit_delayed': ld, 'legit_warned': lw,
            'prec_delay': prec_d, 'prec_warn': prec_w,
        })

results_cal_df = pd.DataFrame(results_cal)
print(f"\nConfiguraciones válidas (calibrado): {len(results_cal_df)}")

# ---- 4. Comparación contra la versión sin calibrar ----
if len(results_cal_df) == 0:
    print("\n!! Ninguna configuración pasó los guardrails. Ampliar las rejillas.")
else:
    best_cal = results_cal_df.nlargest(1, 'valor_neto').iloc[0]

    actions_cal = np.where(el >= best_cal['delay_t'], 'delay',
                  np.where(el >= best_cal['warn_t'], 'scam_alert', 'let_through'))
    eval_cal = evaluate_policy(actions_cal, y_test, 'Proposed CALIBRATED')

    print(f"\n=== Umbrales óptimos calibrados ===")
    print(f"  warn  = ${best_cal['warn_t']:,.0f}   (teórico: ${UMBRAL_WARN_TEO:,.0f})")
    print(f"  delay = ${best_cal['delay_t']:,.0f}   (teórico: ${UMBRAL_DELAY_TEO:,.0f})")

    print(f"\n=== Chequeo de borde de rejilla ===")
    print(f"  warn={best_cal['warn_t']:.0f} en [{warn_grid_cal.min()}, {warn_grid_cal.max()}]"
          f"{'  <-- BORDE!' if best_cal['warn_t'] in (warn_grid_cal.min(), warn_grid_cal.max()) else ''}")
    print(f"  delay={best_cal['delay_t']:.0f} en [{delay_grid_cal.min()}, {delay_grid_cal.max()}]"
          f"{'  <-- BORDE!' if best_cal['delay_t'] in (delay_grid_cal.min(), delay_grid_cal.max()) else ''}")

    print(f"\n=== SIN CALIBRAR  vs  CALIBRADO ===")
    print(f"{'':34s} {'Sin calibrar':>16s} {'Calibrado':>16s} {'Delta':>14s}")
    print("-" * 84)
    for k, lab, fm in [
        ('valor_neto',          'Valor neto (MXN)',      '$'),
        ('loss_prevented_est',  'Pérdida evitada (est)', '$'),
        ('costo',               'Costo de fricción',     '$'),
        ('legit_delayed',       'Legítimos demorados',   'n'),
        ('legit_warned',        'Legítimos advertidos',  'n'),
        ('legit_hours_blocked', 'Horas bloqueadas',      'h'),
        ('ns_mxn_per_hour',     'North Star ($/hr)',     '$'),
        ('recall',              'Recall',                'p'),
        ('recall_eff',          'Recall efectivo',       'p'),
        ('prec_delay',          'Precisión demora',      'p'),
    ]:
        a, b = eval_proposed[k], eval_cal[k]
        d = b - a
        if   fm == '$': f = lambda v: f"${v:,.0f}"
        elif fm == 'h': f = lambda v: f"{v:,.0f} hrs"
        elif fm == 'p': f = lambda v: f"{v:.4f}"
        else:           f = lambda v: f"{v:,.0f}"
        print(f"  {lab:32s} {f(a):>16s} {f(b):>16s} {f(d):>14s}")

    delta = eval_cal['valor_neto'] - eval_proposed['valor_neto']
    print(f"\n>>> VEREDICTO: la calibración "
          f"{'MEJORA' if delta > 0 else 'NO mejora'} el valor neto en ${abs(delta):,.0f} "
          f"({100*delta/abs(eval_proposed['valor_neto']):+.1f}%)")

In [ ]:
%python
for k, v in eval_cal.items():
    print(f"  {k:26s} {v}")

In [ ]:
%python
# ===============================================================
# FIX Error 4: 3-policy comparison on SAME test set
# 1. Proposed model policy (best by valor_neto)
# 2. Current rule-based policy (P-01 to P-05) applied to test rows
# 3. MTU-only baseline (the brief asks for this)
# All use the same effectiveness coefficients.
# ===============================================================

best = results_df.nlargest(1, 'valor_neto').iloc[0]

# ---- Helper: evaluate a policy assignment on test set ----
def evaluate_policy(actions_arr, label, name):
    """Compute standard metrics for a policy action vector on the test set."""
    is_s = label == 1
    loss_v = pdf_test['loss_amount_mxn'].fillna(0)
    n = len(actions_arr)
    
    sd = (actions_arr == 'delay') & is_s
    sw = (actions_arr == 'scam_alert') & is_s
    st = (actions_arr == 'let_through') & is_s
    ld = ((actions_arr == 'delay') & ~is_s).sum()
    lw = ((actions_arr == 'scam_alert') & ~is_s).sum()
    
    nd = (actions_arr == 'delay').sum()
    nw = (actions_arr == 'scam_alert').sum()
    
    loss_exp_d = loss_v[sd].sum()
    loss_exp_w = loss_v[sw].sum()
    loss_thr   = loss_v[st].sum()
    prevented  = E_DELAY * loss_exp_d + E_WARN * loss_exp_w
    costo_val  = ld * (HRS * V_HORA + P_OPS * C_OPS) + lw * C_WARN
    neto       = prevented - costo_val
    
    prec_d = sd.sum() / nd if nd > 0 else 0
    prec_w = sw.sum() / nw if nw > 0 else 0
    lift_d = prec_d / BASE_RATE if BASE_RATE > 0 else 0
    lift_w = prec_w / BASE_RATE if BASE_RATE > 0 else 0
    
    recall = (sd.sum() + sw.sum()) / is_s.sum() if is_s.sum() > 0 else 0
    
    # North Star del reto: MXN evitados por hora de bloqueo a legítimos
    hrs_blocked = ld * HRS
    ns_mxn_per_hour = prevented / hrs_blocked if hrs_blocked > 0 else 0
    # Recall ponderado por la efectividad real de cada acción
    recall_eff = (E_DELAY * sd.sum() + E_WARN * sw.sum()) / is_s.sum() if is_s.sum() > 0 else 0

    return {
        'policy': name,
        'scams_delayed': sd.sum(), 'scams_warned': sw.sum(), 'scams_through': st.sum(),
        'loss_exposure_delayed': loss_exp_d, 'loss_exposure_warned': loss_exp_w,
        'loss_prevented_est': prevented, 'loss_through': loss_thr,
        'legit_delayed': ld, 'legit_warned': lw,
        'legit_hours_blocked': hrs_blocked,
        'ns_mxn_per_hour': ns_mxn_per_hour,
        'recall_eff': recall_eff,
        'prec_delay': prec_d, 'prec_warn': prec_w,
        'lift_delay': lift_d, 'lift_warn': lift_w,
        'recall': recall,
        'costo': costo_val, 'valor_neto': neto,
        'pct_warned': nw / n, 'pct_delayed': nd / n,
    }

# ---- Policy 1: PROPOSED (model-based, expected loss thresholds) ----
warn_t_best  = best['warn_t']
delay_t_best = best['delay_t']
actions_proposed = np.where(pdf_test['expected_loss'] >= delay_t_best, 'delay',
                   np.where(pdf_test['expected_loss'] >= warn_t_best, 'scam_alert', 'let_through'))

# ---- Policy 2: CURRENT (P-01 to P-05 rules applied to test rows) ----
# Replicate the 5 rules from the policy_events descriptions
mtu_r = pdf_test['mtu_ratio'].fillna(0)
amt_t = pdf_test['amount_mxn']
mtu_decl = pdf_test['mtu_declared_mxn'].fillna(1e9)
new_cp = pdf_test['new_counterparty'] == 1
hour = pdf_test['hour_of_day']
ch = pdf_test['channel']

p01 = mtu_r >= 1.00                                          # delay
p02 = (mtu_r >= 0.85) & (mtu_r < 1.00)                       # scam_alert
p03 = amt_t >= 0.50 * mtu_decl                                # scam_alert
p04 = new_cp & (amt_t >= 0.30 * mtu_decl)                     # scam_alert
p05 = (ch == 'cash_out') & (hour >= 0) & (hour <= 5)          # delay

# Priority: delay rules first (P-01, P-05), then warning rules (P-02, P-03, P-04)
actions_current = np.full(len(pdf_test), 'let_through', dtype=object)
actions_current[p02 | p03 | p04] = 'scam_alert'
actions_current[p01 | p05] = 'delay'  # delay overrides warning

# ---- Policy 3: MTU-ONLY baseline (delay if mtu_ratio > 1, warn if > 0.85) ----
actions_mtu = np.full(len(pdf_test), 'let_through', dtype=object)
actions_mtu[(mtu_r >= 0.85) & (mtu_r < 1.00)] = 'scam_alert'
actions_mtu[mtu_r >= 1.00] = 'delay'

# ---- Evaluate all three ----
eval_proposed = evaluate_policy(actions_proposed, y_test, f'Proposed (warn={warn_t_best:.0f}, delay={delay_t_best:.0f} MXN)')
eval_current  = evaluate_policy(actions_current, y_test, 'Current (P-01..P-05)')
eval_mtu      = evaluate_policy(actions_mtu, y_test, 'MTU-only baseline')

comp_df = pd.DataFrame([eval_proposed, eval_current, eval_mtu])

# ---- Print comparison ----
print(f"=== 3-POLICY COMPARISON on test set (n={len(pdf_test):,}, scams={int(y_test.sum())}) ===")
print(f"{'':40s} {'Proposed':>12s} {'Current':>12s} {'MTU-only':>12s}")
print("-" * 80)
for metric in ['scams_delayed', 'scams_warned', 'scams_through', 'recall', 'recall_eff',
               'loss_prevented_est', 'loss_through',
               'legit_delayed', 'legit_warned', 'legit_hours_blocked',
               'ns_mxn_per_hour',
               'prec_delay', 'prec_warn', 'lift_delay', 'lift_warn',
               'pct_delayed', 'pct_warned',
               'costo', 'valor_neto']:
    vals = [comp_df.loc[i, metric] for i in range(3)]
    if metric in ['recall', 'recall_eff', 'prec_delay', 'prec_warn', 'pct_delayed', 'pct_warned']:
        fmt = [f"{v:.4f}" for v in vals]
    elif metric in ['lift_delay', 'lift_warn']:
        fmt = [f"{v:.1f}x" for v in vals]
    elif metric == 'ns_mxn_per_hour':
        fmt = [f"${v:,.2f}/hr" for v in vals]
    elif metric == 'legit_hours_blocked':
        fmt = [f"{v:,.0f} hrs" for v in vals]     # horas, NO pesos
    elif metric in ['loss_prevented_est', 'loss_through', 'costo', 'valor_neto']:
        fmt = [f"${v:,.0f}" for v in vals]
    else:
        fmt = [f"{v:,.0f}" for v in vals]
    print(f"  {metric:38s} {fmt[0]:>14s} {fmt[1]:>14s} {fmt[2]:>14s}")

# ---- Bucket precision vs base rate check ----
print(f"\n=== Acceptance Criteria Check ===")
print(f"Base rate: {BASE_RATE:.5f}")
for pol in [eval_proposed, eval_current, eval_mtu]:
    name = pol['policy'][:20]
    ok_d = pol['prec_delay'] >= BASE_RATE or pol['scams_delayed'] == 0
    ok_w = pol['prec_warn'] >= BASE_RATE or pol['scams_warned'] == 0
    ok_pct = pol['pct_warned'] <= 0.10
    print(f"  {name:20s}: prec_delay={pol['prec_delay']:.4f} {'OK' if ok_d else 'FAIL'}  "
          f"prec_warn={pol['prec_warn']:.4f} {'OK' if ok_w else 'FAIL'}  "
          f"pct_warned={pol['pct_warned']:.4f} {'OK' if ok_pct else 'FAIL'}")

In [ ]:
%python
# ===============================================================
# EXPORT PARA QUICKSIGHT — 3 tablas en formato tidy
# ===============================================================
import pandas as pd, numpy as np, os

OUT = "/Volumes/usr/luisdominguez/hackaton/dashboard"   # <-- ajusta a tu volumen
os.makedirs(OUT, exist_ok=True)

# ---------------------------------------------------------------
# TABLA 1 — Comparación de políticas (formato largo, ideal QuickSight)
# ---------------------------------------------------------------
politicas = {
    "Propuesta (modelo calibrado)": eval_cal if 'eval_cal' in dir() else eval_proposed,
    "Actual (P-01 a P-05)":         eval_current,
    "Baseline solo-MTU":            eval_mtu,
}

catalogo = [
    # (clave, etiqueta, grupo, unidad, mayor_es_mejor)
    ('scams_delayed',       'Estafas demoradas',            'Deteccion', 'conteo', 1),
    ('scams_warned',        'Estafas advertidas',           'Deteccion', 'conteo', 1),
    ('scams_through',       'Estafas no detectadas',        'Deteccion', 'conteo', 0),
    ('recall',              'Recall',                       'Deteccion', 'ratio',  1),
    ('recall_eff',          'Recall efectivo',              'Deteccion', 'ratio',  1),
    ('loss_prevented_est',  'Perdida evitada (est)',        'Economico', 'MXN',    1),
    ('loss_through',        'Perdida no evitada',           'Economico', 'MXN',    0),
    ('costo',               'Costo de friccion',            'Economico', 'MXN',    0),
    ('valor_neto',          'Valor neto',                   'Economico', 'MXN',    1),
    ('ns_mxn_per_hour',     'North Star (MXN por hora)',    'Economico', 'MXN',    1),
    ('legit_delayed',       'Clientes legitimos demorados', 'Friccion',  'conteo', 0),
    ('legit_warned',        'Clientes legitimos advertidos','Friccion',  'conteo', 0),
    ('legit_hours_blocked', 'Horas bloqueadas a legitimos', 'Friccion',  'horas',  0),
    ('prec_delay',          'Precision zona demora',        'Calidad',   'ratio',  1),
    ('prec_warn',           'Precision zona advertencia',   'Calidad',   'ratio',  1),
    ('lift_delay',          'Lift zona demora',             'Calidad',   'veces',  1),
    ('lift_warn',           'Lift zona advertencia',        'Calidad',   'veces',  1),
    ('pct_delayed',         'Pct transacciones demoradas',  'Calidad',   'pct',    0),
    ('pct_warned',          'Pct transacciones advertidas', 'Calidad',   'pct',    0),
]

filas = []
for nombre_pol, ev in politicas.items():
    for clave, etiqueta, grupo, unidad, mejor in catalogo:
        filas.append({
            'politica': nombre_pol,
            'es_propuesta': int(nombre_pol.startswith('Propuesta')),
            'grupo_metrica': grupo,
            'metrica': etiqueta,
            'metrica_key': clave,
            'valor': float(ev[clave]),
            'unidad': unidad,
            'mayor_es_mejor': mejor,
        })

t1 = pd.DataFrame(filas)

# Delta vs politica actual, para las tarjetas de KPI
base = t1[t1.politica == 'Actual (P-01 a P-05)'].set_index('metrica_key')['valor']
t1['valor_politica_actual'] = t1['metrica_key'].map(base)
t1['delta_abs'] = t1['valor'] - t1['valor_politica_actual']
t1['delta_pct'] = np.where(t1['valor_politica_actual'] != 0,
                           100 * t1['delta_abs'] / t1['valor_politica_actual'].abs(), np.nan)
t1['veces_vs_actual'] = np.where(t1['valor_politica_actual'] != 0,
                                 t1['valor'] / t1['valor_politica_actual'], np.nan)

t1.to_csv(f"{OUT}/dash_comparacion_politicas.csv", index=False)
print(f"TABLA 1 -> {len(t1)} filas | dash_comparacion_politicas.csv")

# ---------------------------------------------------------------
# TABLA 2 — Diagnostico de la politica actual, regla por regla
# ---------------------------------------------------------------
pe = spark.table("policy_events").toPandas()
sr = spark.table("scam_reports").toPandas()[['txn_id', 'confirmed_scam', 'loss_amount_mxn']]
pe = pe.merge(sr, on='txn_id', how='left')
pe['is_scam'] = pe['confirmed_scam'].fillna(False).astype(bool)

BASE_GLOBAL = 2872 / 901286   # tasa base del dataset completo

t2 = (pe.groupby(['rule_id', 'rule_description'])
        .apply(lambda g: pd.Series({
            'disparos':        len(g),
            'demoras':         (g.action_taken == 'delay').sum(),
            'advertencias':    (g.action_taken == 'scam_alert').sum(),
            'holdout':         (g.action_taken == 'none').sum(),
            'estafas':         g.is_scam.sum(),
            'exposicion_mxn':  g.loc[g.is_scam, 'loss_amount_mxn'].sum(),
            'minutos_bloqueados': g.minutes_blocked.sum(),
            'horas_bloqueadas':   g.minutes_blocked.sum() / 60.0,
            'contactos_ops':   (g.ops_contact_flag == True).sum(),
        }))
        .reset_index())

t2['accion_principal'] = np.where(t2.demoras > t2.advertencias, 'Demora', 'Advertencia')
t2['precision']  = t2.estafas / t2.disparos
t2['lift']       = t2.precision / BASE_GLOBAL
t2['peor_que_azar'] = (t2.lift < 1).astype(int)
t2['mxn_por_hora_bloqueada'] = np.where(t2.horas_bloqueadas > 0,
                                        t2.exposicion_mxn / t2.horas_bloqueadas, np.nan)

t2.to_csv(f"{OUT}/dash_diagnostico_reglas.csv", index=False)
print(f"TABLA 2 -> {len(t2)} filas | dash_diagnostico_reglas.csv")

# ---------------------------------------------------------------
# TABLA 3 — Evolucion semanal (el cambio de regimen)
# ---------------------------------------------------------------
t3 = (pdf.groupby('txn_week')
        .apply(lambda g: pd.Series({
            'transacciones':      len(g),
            'estafas':            g.label.sum(),
            'tasa_estafa':        g.label.mean(),
            'perdida_total_mxn':  g.loc[g.label == 1, 'loss_amount_mxn'].sum(),
            'perdida_prom_mxn':   g.loc[g.label == 1, 'loss_amount_mxn'].mean(),
            'pct_contraparte_nueva': g.loc[g.label == 1, 'new_counterparty'].mean(),
            'pct_dispositivo_nuevo': g.loc[g.label == 1, 'new_device'].mean(),
            'hora_promedio':         g.loc[g.label == 1, 'hour_of_day'].mean(),
            'monto_promedio_mxn':    g.loc[g.label == 1, 'amount_mxn'].mean(),
        }))
        .reset_index())

t3['periodo']  = np.where(t3.txn_week >= 24, 'Brote (sem 24-27)', 'Estable (sem 9-23)')
t3['conjunto'] = np.where(t3.txn_week >= 24, 'Test', 'Entrenamiento')
t3 = t3[t3.estafas > 0]

t3.to_csv(f"{OUT}/dash_evolucion_semanal.csv", index=False)
print(f"TABLA 3 -> {len(t3)} filas | dash_evolucion_semanal.csv")

print(f"\nArchivos listos en {OUT}")
display(t1.head(20))
display(t2)
display(t3)

%md
### Summary — Full Dataset Results (Final)

All figures use **observed effectiveness coefficients** (E\_delay=0.758, E\_warn=0.215), a **temporal train/test split** (weeks ≤23 / ≥24), a **calibrated fraud score**, and the **same test denominator** (158,840 txns, 1,249 scams) for all policies. Loss figures are *estimates*, not raw exposure.

---

#### Point 1 — Current Policy Diagnosis (full dataset)

| Rule | Triggered | Action | Scams | Loss Exposure | Min Blocked | Ops Contacts |
|---|---|---|---|---|---|---|
| P-01 | 20,685 | delay | 41 | $300,673 | 11,232,671 | 6,607 |
| P-02 | 9,802 | scam_alert | 68 | $335,451 | 0 | 0 |
| P-03 | 1,320 | scam_alert | 9 | $168,759 | 0 | 0 |
| P-04 | 806 | scam_alert | 10 | $94,161 | 0 | 0 |
| P-05 | 5,312 | delay | 7 | $4,829 | 2,862,777 | 1,734 |
| **Total** | **37,925** | | **135** | **$903,873** | **14,095,448** | **8,341** |

Catch rate: **4.7%** of confirmed scams (135 / 2,872). Loss exposure touched: **11.8%**.

**Precision inversion — the core diagnosis.** Against the global base rate (0.319%):

| Rule | Precision | Lift | Action |
|---|---|---|---|
| P-04 | 1.241% | **3.89x** | warning |
| P-02 | 0.694% | 2.18x | warning |
| P-03 | 0.682% | 2.14x | warning |
| P-01 | 0.198% | **0.62x** | **delay** |
| P-05 | 0.132% | **0.41x** | **delay** |

The three most precise rules only warn. The two **below random** impose the 12-hour block. Action severity is ordered by MTU consumption, not by fraud probability.

**Caveat on "loss exposure":** every confirmed scam has `completed_flag = true`, so this is money that **was lost** on transactions the policy fired on — not money saved. Prevented fraud is unobservable by construction.

**Holdout counterfactual — the only causal estimate:** 12 scams in 2,320 holdout txns (0.517%) vs 123 in 35,605 treated (0.345%). Implies ~33% relative reduction, ~61 scams and ~$410K prevented. With 12 events the confidence interval includes no effect — directional, not conclusive.

---

#### Point 2 — Fraud Model

**GBM** (sklearn, 300 trees, depth 4, class-weighted 456:1) with **temporal split**.

| Metric | Train (wk ≤23) | Test (wk ≥24) |
|---|---|---|
| AUC-ROC | 0.9266 | **0.9458** |
| AUC-PR | — | **0.3939** |
| Rows | 742,446 | 158,840 |
| Scams | 1,623 | 1,249 |
| Base rate | 0.219% | 0.786% |

Trained **only on pre-shift data** (weeks ≤23), the model achieves AUC-ROC 0.9458 and recall 0.862 on the shift period — it detects the new pattern without having seen it.

**Feature Importance (top 7):**

| Feature | Importance |
|---|---|
| `new_counterparty` | 0.330 |
| `new_device` | 0.128 |
| `tenure_months` | 0.124 |
| `ticket_ratio` | 0.096 |
| `mtu_gap_ratio` | 0.059 |
| `mtu_ratio` | 0.050 |
| `amount_mxn` | 0.046 |

`mtu_ratio` — the signal the entire current policy is built on — ranks **6th**. Behavioural novelty (`new_counterparty` + `new_device`) carries 4x more weight than the MTU features combined.

**Leakage check:** `prior_scam` scam rate stable across week buckets (0.0070 / 0.0068 / 0.0080). Model without `prior_scam` + `tenure_months`: AUC-ROC 0.9358 vs 0.9458. Leakage not a concern.

---

#### Point 3 — Proposed Policy (calibrated expected loss)

##### Score calibration

The 456:1 class weighting inflates predicted probabilities. Corrected analytically via the odds relation: `p_calibrated = (odds_raw / 456) / (1 + odds_raw / 456)`.

| | Value |
|---|---|
| Mean raw score | 0.26361 |
| Mean calibrated score | 0.00180 |
| Train prevalence | 0.00219 |
| Test prevalence | 0.00786 |

The calibrated mean lands on the **training** prevalence, as it should — the model can only know the regime it was trained on. The **4.4x gap** between what the model expects (0.18%) and what actually happened (0.79%) *is the regime shift, quantified*. See Point 4.

##### Decision rule

`risk_mxn = calibrated_score × amount_mxn`

| Risk (MXN) | Action |
|---|---|
| < 6 | allow |
| 6 – 30 | warn |
| ≥ 30 | delay |

**Structural guarantee:** since `score ≤ 1`, no transaction below $30 can ever be delayed and none below $6 can be warned, regardless of model output.

##### Thresholds: theory and data agree

The economically optimal delay threshold is derivable from first principles: delay when `(E_delay − E_warn) × expected_loss > HRS×V_HORA + P_OPS×C_OPS`, i.e. `0.543 × EL > 9×10 + 0.34×50 = 107`, giving **EL > $197**.

Adjusting for the 4.37x calibration gap, that corresponds to **$45 in calibrated units**. The grid search independently found **$30** — same order of magnitude with a $10-step grid. Economic theory and empirical optimisation converge.

##### 3-Policy Comparison (test set: 158,840 txns, 1,249 scams, base rate 0.786%)

| Metric | Proposed (calibrated) | Current (P-01..P-05) | MTU-only |
|---|---|---|---|
| Scams delayed | **833** | 27 | 22 |
| Scams warned | 244 | 59 | 56 |
| Scams let through | **172** | 1,163 | 1,171 |
| **Recall** | **0.862** | 0.069 | 0.062 |
| Recall (effectiveness-weighted) | **0.548** | 0.027 | 0.023 |
| **Loss prevented (est)** | **$2,941,687** | $175,647 | $168,526 |
| Loss let through | **$200,906** | $4,041,438 | $4,066,956 |
| Legit delayed | **2,506** | 5,190 | 4,267 |
| Legit warned | 11,830 | 2,503 | 2,221 |
| **Legit hours blocked** | **22,554 hrs** | 46,710 hrs | 38,403 hrs |
| **North Star ($ prevented / hr blocked)** | **$130.43/hr** | $3.76/hr | $4.39/hr |
| Prec (delay bucket) | **0.249** | 0.005 | 0.005 |
| Prec (warn bucket) | 0.020 | 0.023 | 0.025 |
| **Lift (delay bucket)** | **31.7x** | **0.7x** | **0.7x** |
| Lift (warn bucket) | 2.6x | 2.9x | 3.1x |
| % transactions delayed | 2.1% | 3.3% | 2.7% |
| % transactions warned | 7.6% | 1.6% | 1.4% |
| Friction cost (MXN) | **$291,802** | $560,336 | $461,011 |
| **Net value (MXN)** | **+$2,649,885** | **−$384,689** | **−$292,485** |

##### Headline

* **16.7x more loss prevented** than the policy in production.
* **34.7x better North Star** ($130.43 vs $3.76 per hour of legitimate-customer blocking).
* **52% fewer blocked hours** (22,554 vs 46,710) — the challenge's guardrail improves, it does not degrade.
* The current policy and the MTU-only baseline both have **negative net value**: the friction they impose costs more than the fraud they prevent.

The proposal delays **half as many** transactions as the current policy while catching **30x more scams**. This is not a trade-off — it is a reallocation of the same friction budget toward transactions where the money at risk justifies it.

The only metric that rises is warnings (2,503 → 11,830), which cost no blocked time. Warning volume stays at 7.6% of transactions, under the 10% ceiling.

##### Effect of calibration

| Metric | Uncalibrated | Calibrated | Δ |
|---|---|---|---|
| Net value | $2,259,625 | **$2,649,885** | **+$390,261 (+17.3%)** |
| Loss prevented | $2,885,295 | $2,941,687 | +$56,393 |
| Friction cost | $625,670 | **$291,802** | **−$333,868** |
| Legit delayed | 5,586 | **2,506** | **−55%** |
| Hours blocked | 50,274 | **22,554** | **−55%** |
| Delay-bucket precision | 0.109 | **0.249** | **2.3x** |
| North Star | $57.39/hr | **$130.43/hr** | **+127%** |

The raw score compresses the upper tail (a 0.99 reads as only 1.6x a 0.62, when the true risk ratio is ~50x), letting large amounts substitute for certainty. Calibrating restores the correct weighting and reallocates delays toward near-certain cases: **fewer delays, more scams caught**.

**Acceptance criteria** (base rate 0.786%, test-set prevalence):

* Proposed: prec\_delay=0.249 OK, prec\_warn=0.020 OK, pct\_warned=0.076 OK. **All pass.**
* Current (P-01..P-05): prec\_delay=0.005 **FAIL** — the delay bucket is worse than random (lift 0.7x). It applies the harshest action to a population *safer* than average.
* MTU-only: prec\_delay=0.005 **FAIL**. Same pathology.
* Optimum not at grid edge: warn=6 (range 2–298), delay=30 (range 20–1,990).
* Current-policy replication validated: reconstructed rules fire on 4.9% of test transactions vs 4.2% observed in `policy_events`.

---

#### Point 4 — Emerging Pattern

Weeks 24–27 concentrate **44.5% of all confirmed scams and 59% of all losses** in 4 of ~20 weeks.

| Metric | Weeks 9–23 | Weeks 24–27 |
|---|---|---|
| Avg loss | ~$2,000 | **$3,362–$3,671** |
| New counterparty rate | ~0.50 | **0.86–0.88** |
| New device rate | ~0.32 | **0.43–0.47** |
| Avg hour | ~12.3 | **17.3–17.9** |
| Weekly volume | ~110 | **314–425** |

The signature is **behavioural, not volumetric**: near-universal use of never-before-seen counterparties, shifted to evening hours, at nearly double the average ticket. MTU rules are static volume thresholds and are structurally blind to it — they catch **6.9%** on these weeks, with their delay bucket performing below random.

##### Calibration gap as a regime-change detector

The calibrated score averages 0.180% on weeks 24+ while realised prevalence is 0.786% — a **4.4x under-prediction**. This is not a model defect: the model faithfully reproduces the regime it was trained on.

It is, however, an **operational monitoring signal**. When calibrated predictions systematically fall below realised fraud rates, the environment has shifted. This requires no new labels, no retraining and no analyst review — it is a continuously computable alarm that the current MTU policy has no equivalent of.

##### Open line of investigation

`counterparty_id` remains unexploited. The natural hypothesis for the outbreak is **shared mule accounts** — multiple victims paying the same counterparties. No current rule inspects this field. Any counterparty-history feature must be built on a strictly causal window, since `reported_ts` lags the transaction.

---

#### Business Parameters

| Parameter | Value | Source |
|---|---|---|
| E\_delay | 0.758 | 1 − bypass\_granted rate, observed |
| E\_warn | 0.215 | 1 − customer\_proceeded rate, observed |
| Avg hours blocked | 9.0 | 543 min observed mean (not the nominal 12h — bypasses cut it short) |
| P(ops contact \| delay) | 0.34 | ops\_contact\_flag rate, observed |
| Base rate | `y_test.mean()` = 0.00786 | Computed, never hardcoded |
| Cost per blocked hour | 10.0 MXN | **Product assumption — to be confirmed** |
| Cost per warning | 2.0 MXN | **Product assumption — to be confirmed** |
| Cost per ops contact | 50.0 MXN | **Product assumption — to be confirmed** |

The three cost parameters are business judgments, not data. Reference point: the current policy delivers **$3.76** of prevented loss per hour of legitimate-customer blocking — if the business values an hour above that, **the policy in production destroys net value**.

With a calibrated score the delay threshold becomes directly derivable — `(HRS×V_HORA + P_OPS×C_OPS) / (E_delay − E_warn)` — so Product can recompute it from a changed cost assumption without re-running the optimisation.

---

#### Documented Assumptions & Limitations

1. **MTU is treated as a declared monthly transactional ceiling.** Observed `mtu_ratio` reaches 5.25, so it operates as a monitoring threshold that triggers policy, not as a hard cap.
2. **Loss equals transaction amount** for confirmed scams (avg amount = avg loss = $2,675.38), so expected loss = score × amount with no separate severity model.
3. **Post-action columns** (`customer_proceeded`, `bypass_*`, `ops_contact_flag`, `minutes_blocked`) are excluded from features but used to calibrate action effectiveness — they inform the cost function, never the model.
4. **Censored population:** successfully prevented scams never become reports, so the label undercounts. Measured precision is a lower bound.
5. **Calibration is anchored to the training regime.** The 4.4x gap on weeks 24+ means absolute peso figures are conservative; the ranking correction — which is what the policy depends on — holds regardless.
6. **The causal estimate rests on 12 holdout events.** Wide interval.
7. **MTU regulatory compliance (Art. 287 Bis) is a separate layer** from fraud prevention and should be reported separately, not counted as fraud caught.